# Partial Differential Equations: The (Parabolic) Heat Equation

---
REFS:
- Chapra & Canale, *Numerical Methods for Engineers*, ch. 30
- Barba et al., [Numerical MOOC: "Spreading out"](https://github.com/numerical-mooc/numerical-mooc/tree/master/lessons/04_spreadout)
- [Solving PDEs in Python (MOOC book)](https://aquaulb.github.io/book_solving_pde_mooc/solving_pde_mooc/notebooks/01_Introduction/01_00_Preface.html)
- Previous notebook: [Finite differences and relaxation](FiniteDifferences-Relaxation.ipynb)
---

```{admonition} Learning goals
:class: tip
After this notebook you should be able to
1. discretize the heat equation with the explicit FTCS scheme and run it;
2. derive and *observe* its stability limit $\lambda = \alpha\Delta t/h^2 \le 1/2$;
3. validate a solver against exact solutions and measure its order of accuracy;
4. implement insulated (Neumann) boundaries and source/sink terms;
5. use implicit methods (backward Euler and Crank–Nicolson) through one sparse-matrix "θ-method", and know when they pay off;
6. solve the heat equation in two dimensions and connect it with relaxation.
```

In [ ]:
# Setup: all imports and small helpers used in the notebook
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
import scipy.sparse as sp
import scipy.sparse.linalg as spla
from IPython.display import HTML, Image, display

sys.path.insert(0, os.path.abspath(os.path.join("..", "..", "utils")))
from embed import embed          # shared helper to embed videos and web pages


def save_example(name):
    """Save the current figure as fig/<name>.png, the reference result shown after an exercise.

    Only the instructor copy (next to nbgrader_config.py) saves, so the reference
    figures are never overwritten by an unfinished exercise."""
    if os.path.exists(os.path.join("..", "..", "nbgrader_config.py")):
        plt.savefig(f"fig/{name}.png", dpi=120)

os.makedirs("fig", exist_ok=True)


def animate_1d(x, frames, times, ylabel="$T$", interval=100):
    """Animate a list of 1D profiles u(x) at the given times."""
    fig, ax = plt.subplots(figsize=(6, 3.5))
    line, = ax.plot(x, frames[0], lw=2)
    lo, hi = min(f.min() for f in frames), max(f.max() for f in frames)
    pad = 0.05*(hi - lo) + 1e-12
    ax.set(xlim=(x[0], x[-1]), ylim=(lo - pad, hi + pad), xlabel="$x$", ylabel=ylabel)
    title = ax.set_title("")

    def update(k):
        line.set_ydata(frames[k])
        title.set_text(f"t = {times[k]:.3g}")
        return line, title

    ani = animation.FuncAnimation(fig, update, frames=len(frames), interval=interval)
    plt.close(fig)
    return HTML(ani.to_jshtml())


def animate_2d(frames, times, cmap="inferno", interval=100):
    """Animate a list of 2D fields u[i, j] (i along x) on the unit square."""
    fig, ax = plt.subplots(figsize=(4.8, 4))
    vmin, vmax = min(f.min() for f in frames), max(f.max() for f in frames)
    im = ax.imshow(frames[0].T, origin="lower", extent=(0, 1, 0, 1), cmap=cmap, vmin=vmin, vmax=vmax)
    plt.colorbar(im, ax=ax, label="$T$")
    ax.set(xlabel="$x$", ylabel="$y$")
    title = ax.set_title("")

    def update(k):
        im.set_data(frames[k].T)
        title.set_text(f"t = {times[k]:.3g}")
        return im, title

    ani = animation.FuncAnimation(fig, update, frames=len(frames), interval=interval)
    plt.close(fig)
    return HTML(ani.to_jshtml())

## Diffusion

The [heat equation](https://en.wikipedia.org/wiki/Heat_equation)

$$
\frac{\partial u}{\partial t} = \alpha \nabla^2 u
$$

describes how a temperature (or a concentration, in a diffusion problem) spreads out. The diffusivity $\alpha = k/(\rho c_p)$ combines the thermal conductivity, density and specific heat of the material. It is the typical **parabolic** equation: given an initial condition and boundary conditions, the solution evolves irreversibly towards a steady state, which solves the Laplace equation of the previous notebook.

Two facts set the scales of every heat problem:
- a structure of size $L$ needs a time of order $L^2/\alpha$ to reach equilibrium (doubling the size takes four times longer);
- a point-like initial heat pulse spreads as a Gaussian whose variance grows linearly, $\sigma^2(t) = \sigma_0^2 + 2\alpha t$.

In [ ]:
embed("https://visualpde.com/sim/?preset=heatEquation1D")

## The explicit FTCS method

Discretize space with $x_i = i h$ and time with $t_n = n\,\Delta t$, and write $u_i^n \approx u(x_i, t_n)$. Use a **forward** difference in time (first order) and a **centered** second difference in space (second order):

$$
\frac{u^{n+1}_i - u^n_i}{\Delta t} = \alpha\, \frac{u_{i+1}^n - 2u_i^n + u_{i-1}^n}{h^2}
\quad\Longrightarrow\quad
\boxed{u_{i}^{n+1} = u_{i}^{n}+\lambda\left(u_{i+1}^{n}-2u_{i}^{n}+u_{i-1}^{n}\right), \qquad \lambda = \frac{\alpha\,\Delta t}{h^2}.}
$$

This is the [FTCS (Forward Time, Centered Space)](https://en.wikipedia.org/wiki/FTCS_scheme) scheme. It is **explicit**: each new value is computed directly from the old ones, with no system of equations to solve.

```{figure} https://upload.wikimedia.org/wikipedia/commons/c/c2/Explicit_method-stencil.svg
:width: 35%
:align: center
FTCS stencil: three old values give one new value.
```

In NumPy one FTCS step is a single line, just like a Jacobi sweep. The boundary values are not touched, so Dirichlet conditions are kept automatically.

In [ ]:
def ftcs(u, lam):
    """One FTCS step of the 1D heat equation. Returns a new array; the end values are copied unchanged."""
    u_new = u.copy()
    u_new[1:-1] = u[1:-1] + lam*(u[2:] - 2*u[1:-1] + u[:-2])
    return u_new

### Example: a thin rod (Chapra, example 30.1)

A rod of length $L = 10$ cm with $\alpha = 0.835$ cm²/s starts at $T = 0$ °C, and its ends are suddenly held at $T(0) = 100$ °C and $T(10) = 50$ °C. With $h = 2$ cm and $\Delta t = 0.1$ s we get $\lambda = 0.835\times0.1/2^2 = 0.020875$. The first two steps can be checked by hand, and they match the values in Chapra's table ($2.0875$ and $1.04375$ at $t = 0.1$ s).

In [ ]:
L, h, dt, alpha = 10.0, 2.0, 0.1, 0.835
x = np.arange(0.0, L + h/2, h)
lam = alpha*dt/h**2
u = np.zeros_like(x)
u[0], u[-1] = 100.0, 50.0
print(f"lambda = {lam}")
print("t [s]" + "".join(f"   x={xi:<4.0f}" for xi in x))
for n in range(1, 3):
    u = ftcs(u, lam)
    print(f"{n*dt:5.1f}" + "".join(f"{ui:10.4f}" for ui in u))

Running long enough, the profile approaches the linear steady state $T = 100 - 5x$, the solution of $T'' = 0$. The time scale is $L^2/\alpha \approx 120$ s. A finer grid ($h = 0.5$ cm) gives smoother curves; note that $\Delta t$ must then shrink with $h^2$ to keep the same $\lambda$ (see {ref}`heat-stability`).

In [ ]:
L, alpha = 10.0, 0.835
N = 21
x = np.linspace(0, L, N)
h = x[1] - x[0]
lam = 0.4
dt = lam*h**2/alpha
u = np.zeros(N)
u[0], u[-1] = 100.0, 50.0
fig, ax = plt.subplots(figsize=(6, 4))
t, n = 0.0, 0
for t_show in [1, 10, 30, 60, 120, 300]:
    while t < t_show - 1e-12:
        u = ftcs(u, lam)
        t += dt
    ax.plot(x, u, "o-", ms=3, label=f"t = {t_show} s")
ax.plot(x, 100 - 5*x, "k--", lw=1, label="steady state $100-5x$")
ax.set(xlabel="$x$ [cm]", ylabel="$T$ [°C]", title=f"Thin rod, FTCS ($\\lambda$={lam}, $\\Delta t$={dt:.3f} s)")
ax.legend(fontsize=8)
plt.show()

In [ ]:
# The same evolution as an animation (40 frames)
u = np.zeros(N)
u[0], u[-1] = 100.0, 50.0
frames, times = [u.copy()], [0.0]
nsteps = int(300/dt)
for n in range(1, nsteps + 1):
    u = ftcs(u, lam)
    if n % (nsteps//40) == 0:
        frames.append(u.copy())
        times.append(n*dt)
animate_1d(x, frames, times, ylabel="$T$ [°C]")

(heat-stability)=
## Stability: why $\lambda \le 1/2$

### Von Neumann analysis

Errors (round-off, for example) are combinations of Fourier modes. Insert one mode, $u_i^n = G^n e^{\mathrm{i} k x_i}$, into the FTCS formula. Since $e^{\mathrm{i}kh} + e^{-\mathrm{i}kh} - 2 = -4\sin^2(kh/2)$, each step multiplies the mode by the **amplification factor**

$$
G(k) = 1 - 4\lambda \sin^2\!\left(\frac{kh}{2}\right).
$$

The scheme is stable if no mode grows, $|G| \le 1$ for all $k$. The worst case is the shortest wave the grid can hold, the "saw-tooth" $kh = \pi$ (values $+,-,+,-,\dots$), with $G = 1 - 4\lambda$. Then $|G| \le 1$ requires

$$
\boxed{\lambda = \frac{\alpha\,\Delta t}{h^2} \le \frac{1}{2}.}
$$

In $d$ dimensions with equal spacing the limit becomes $\lambda \le 1/(2d)$, so $\lambda \le 1/4$ in 2D. The price is high: halving $h$ forces $\Delta t$ to drop by 4, so the total cost of a 1D simulation grows 8 times.

### Seeing the instability

Start from the smooth mode $u = \sin(\pi x)$ and track the largest value. For $\lambda > 1/2$ the tiny saw-tooth part created by round-off errors ($\sim 10^{-16}$) is amplified by $|1 - 4\lambda|$ every step until it dominates.

In [ ]:
N = 41
x = np.linspace(0, 1, N)
lams = [0.45, 0.50, 0.55]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
for lam in lams:
    u = np.sin(np.pi*x)
    amp = []
    for n in range(400):
        u = ftcs(u, lam)
        amp.append(np.max(np.abs(u)))
        if lam == 0.55 and n == 219:
            ax2.plot(x, u, "o-", ms=3, color="C2", label=f"$\\lambda$={lam}, step {n+1}")
    ax1.semilogy(amp, label=f"$\\lambda$ = {lam}")
ax1.set(xlabel="step", ylabel="max $|u|$", title="Growth of the largest value")
ax1.legend()
ax2.set(xlabel="$x$", ylabel="$u$", title="The unstable mode is a saw-tooth")
ax2.legend()
plt.tight_layout()
plt.show()

```{exercise} Reading the instability
:label: ex-heat-stability
1. For $\lambda = 0.55$, what growth factor per step does the von Neumann analysis predict for the saw-tooth mode? Check it against the slope of the green curve in the left plot (hint: on a semilog plot, the curve rises by a factor $|G|^{100}$ every 100 steps).
2. Starting from round-off errors of size $10^{-16}$, after about how many steps does the error reach size 1? Compare with the plot.
3. For $\alpha = 1$ and $h = 10^{-3}$, what is the largest stable $\Delta t$, and how many steps are needed to reach $t = 1$?

```

```{admonition} Relaxation is diffusion in disguise
:class: note
In 2D, FTCS with $\lambda = 1/4$ reads $u_{i,j}^{n+1} = u_{i,j}^n + \frac14\left(u_{i+1,j}^n + u_{i-1,j}^n + u_{i,j+1}^n + u_{i,j-1}^n - 4u_{i,j}^n\right) = \frac14\left(u_{i+1,j}^n + u_{i-1,j}^n + u_{i,j+1}^n + u_{i,j-1}^n\right)$, which is exactly a **Jacobi sweep**. Relaxation simulates heat diffusion with the largest stable time step until the steady state (the Laplace solution) is reached. This explains why it needs $\sim N^2$ sweeps: that is the diffusion time $L^2/\alpha$ measured in time steps.
```

## Validation: an exact solution and the order of accuracy

On $[0, 1]$ with $u = 0$ at both ends, the initial condition $u(x, 0) = \sin(\pi x)$ keeps its shape and decays exponentially:

$$
u(x, t) = \sin(\pi x)\, e^{-\alpha\pi^2 t}.
$$

We solve up to $t = 0.1$ with fixed $\lambda = 0.4$, so $\Delta t \propto h^2$, and measure the maximum error. FTCS has error $O(\Delta t) + O(h^2) = O(h^2)$ here.

In [ ]:
alpha, t_end, lam = 1.0, 0.1, 0.4
Ns = np.array([11, 21, 41, 81, 161])
errors = []
for N in Ns:
    x = np.linspace(0, 1, N)
    h = x[1] - x[0]
    nsteps = int(np.ceil(t_end/(lam*h**2/alpha)))
    dt = t_end/nsteps                      # land exactly on t_end
    u = np.sin(np.pi*x)
    for n in range(nsteps):
        u = ftcs(u, alpha*dt/h**2)
    errors.append(np.max(np.abs(u - np.sin(np.pi*x)*np.exp(-alpha*np.pi**2*t_end))))
hs = 1/(Ns - 1)
slope = np.polyfit(np.log(hs), np.log(errors), 1)[0]
fig, ax = plt.subplots(figsize=(5.5, 4))
ax.loglog(hs, errors, "o-", label=f"FTCS (slope {slope:.2f})")
ax.loglog(hs, errors[0]*(hs/hs[0])**2, "k--", lw=1, label=r"$\propto h^2$")
ax.set(xlabel="$h$", ylabel="max error at $t=0.1$", title=r"FTCS error with fixed $\lambda=0.4$")
ax.legend()
plt.show()

## Insulated boundaries (Neumann conditions)

An insulated end lets no heat through: $\partial u/\partial x = 0$. Add a *ghost point* $u_{-1}$ outside the rod. The centered condition $(u_1 - u_{-1})/(2h) = 0$ gives $u_{-1} = u_1$ (second order, better than copying the neighbour), and the FTCS formula at $i = 0$ becomes

$$
u_0^{n+1} = u_0^n + \lambda\left(u_1^n - 2u_0^n + u_{-1}^n\right) = u_0^n + 2\lambda\left(u_1^n - u_0^n\right),
$$

and similarly at the right end. For a prescribed heat flux $\partial u/\partial x = g$, the ghost value is $u_{-1} = u_1 - 2hg$.

```{exercise} An insulated rod
:label: ex-heat-insulated
1. Complete `ftcs_insulated(u, lam)` so that both ends are insulated (two lines, using the formula above).
2. Run the next cell. It starts with $u = 100$ on the left half of a rod $[0, 1]$ and $u = 0$ on the right half ($N = 51$, $\alpha = 1$, $\lambda = 0.4$), plots the profile at several times, and follows the total heat $Q = h\left(\frac{u_0}{2} + u_1 + \dots + u_{N-2} + \frac{u_{N-1}}{2}\right)$ (trapezoidal rule). Is $Q$ conserved? Which value does $u$ tend to, and why?
```

In [ ]:
def ftcs_insulated(u, lam):
    """FTCS step with insulated (zero-derivative) ends, using ghost points."""
    u_new = ftcs(u, lam)
    # YOUR CODE HERE
    pass
    return u_new

In [ ]:
# Given: run it after completing ftcs_insulated
N, alpha, lam = 51, 1.0, 0.4
x = np.linspace(0, 1, N)
h = x[1] - x[0]
dt = lam*h**2/alpha
u = np.where(x < 0.5, 100.0, 0.0)

def total_heat(u):
    return h*(u.sum() - 0.5*(u[0] + u[-1]))

Q0 = total_heat(u)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(x, u, label="t = 0")
t, Q, ts = 0.0, [], []
for t_show in [0.01, 0.03, 0.1, 0.3]:
    while t < t_show - 1e-12:
        u = ftcs_insulated(u, lam)
        t += dt
        Q.append(total_heat(u))
        ts.append(t)
    ax1.plot(x, u, label=f"t = {t_show}")
ax1.axhline(50, color="k", ls="--", lw=1)
ax1.set(xlabel="$x$", ylabel="$u$", title="Rod with insulated ends")
ax1.legend()
ax2.plot(ts, (np.array(Q) - Q0)/Q0)
ax2.set(xlabel="$t$", ylabel="$(Q - Q_0)/Q_0$", title="Relative change of the total heat")
plt.tight_layout()
save_example("heat_insulated")
plt.show()

In [ ]:
for name in ['heat_insulated']:
    if os.path.exists(f"fig/{name}.png"):
        display(Image(filename=f"fig/{name}.png"))

```{exercise} Cooling fin: a sink term
:label: ex-heat-fin
A metal fin loses heat to the surrounding air at temperature $u_\mathrm{env}$ (Newton's law of cooling), which adds a sink term to the equation:

$$
\frac{\partial u}{\partial t} = \alpha \frac{\partial^2 u}{\partial x^2} - \beta\,(u - u_\mathrm{env}).
$$

Take $x \in [0, 1]$, $\alpha = 1$, $\beta = 10$, $u_\mathrm{env} = 20$, the base held at $u(0) = 100$, an **insulated tip** at $x = 1$, and $u = 20$ initially.
1. Complete `fin_step(u, lam, dt, beta, u_env)`: start from `ftcs(u, lam)`, apply the insulated-tip formula at the last point, and subtract `dt*beta*(u - u_env)` from every point except the base (two lines).
2. Run the next cell. It evolves the fin until $t = 1$ ($N = 41$, $\lambda = 0.4$) and compares with the exact steady state
$u(x) = u_\mathrm{env} + (100 - u_\mathrm{env})\,\dfrac{\cosh\big(m(1 - x)\big)}{\cosh m}$, with $m = \sqrt{\beta/\alpha}$. How long does the fin take to reach the steady state?
```

In [ ]:
def fin_step(u, lam, dt, beta, u_env):
    """FTCS step with an insulated tip and Newton cooling."""
    u_new = ftcs(u, lam)
    # YOUR CODE HERE
    pass
    return u_new

In [ ]:
# Given: run it after completing fin_step
N, alpha, beta, u_env, lam = 41, 1.0, 10.0, 20.0, 0.4
x = np.linspace(0, 1, N)
h = x[1] - x[0]
dt = lam*h**2/alpha
u = np.full(N, u_env)
u[0] = 100.0
fig, ax = plt.subplots(figsize=(6, 4))
t = 0.0
for t_show in [0.01, 0.05, 0.1, 1.0]:
    while t < t_show - 1e-12:
        u = fin_step(u, lam, dt, beta, u_env)
        t += dt
    ax.plot(x, u, "o", ms=3, label=f"t = {t_show}")
m = np.sqrt(beta/alpha)
u_steady = u_env + (100 - u_env)*np.cosh(m*(1 - x))/np.cosh(m)
ax.plot(x, u_steady, "k-", lw=1, label="exact steady state")
ax.set(xlabel="$x$", ylabel="$u$", title=r"Cooling fin ($\beta=10$, insulated tip)")
ax.legend()
save_example("heat_fin")
plt.show()
print("max deviation from the exact steady state:", np.max(np.abs(u - u_steady)))

In [ ]:
for name in ['heat_fin']:
    if os.path.exists(f"fig/{name}.png"):
        display(Image(filename=f"fig/{name}.png"))

```{exercise} The diffusion law $\sigma^2 = \sigma_0^2 + 2\alpha t$
:label: ex-heat-diffusion-law
A narrow Gaussian $u(x, 0) = e^{-x^2/(2\sigma_0^2)}$ with $\sigma_0 = 0.3$ spreads in the middle of a long rod $x \in [-5, 5]$ ($u = 0$ at the ends, $N = 201$, $\alpha = 0.5$, $\lambda = 0.4$).
1. Complete `variance(x, u)`, which returns $\sigma^2 = \sum_i x_i^2 u_i / \sum_i u_i$ (one line; the mean is zero by symmetry).
2. Run the next cell. It evolves the Gaussian up to $t = 2$, computes $\sigma^2$ every 50 steps and fits a straight line. Compare the slope with $2\alpha$.
```

In [ ]:
def variance(x, u):
    """Variance of the profile u(x), taking u as a distribution centred at x = 0."""
    # YOUR CODE HERE
    pass

In [ ]:
# Given: run it after completing variance
N, alpha, lam, sigma0 = 201, 0.5, 0.4, 0.3
x = np.linspace(-5, 5, N)
h = x[1] - x[0]
dt = lam*h**2/alpha
u = np.exp(-x**2/(2*sigma0**2))
ts, var = [0.0], [variance(x, u)]
for n in range(1, int(2.0/dt) + 1):
    u = ftcs(u, lam)
    if n % 50 == 0:
        ts.append(n*dt)
        var.append(variance(x, u))
slope, intercept = np.polyfit(ts, var, 1)
fig, ax = plt.subplots(figsize=(5.5, 4))
ax.plot(ts, var, "o", ms=3, label="FTCS")
ax.plot(ts, intercept + slope*np.array(ts), "k-", lw=1, label=f"fit: slope = {slope:.4f} (2$\\alpha$ = {2*alpha})")
ax.set(xlabel="$t$", ylabel=r"$\sigma^2$", title="Spreading of a Gaussian")
ax.legend()
save_example("heat_diffusion_law")
plt.show()

In [ ]:
for name in ['heat_diffusion_law']:
    if os.path.exists(f"fig/{name}.png"):
        display(Image(filename=f"fig/{name}.png"))

## Implicit methods

### The θ-method

The stability limit comes from evaluating the space derivative at the *old* time. Evaluating it (partly) at the *new* time gives a family of schemes. With the tridiagonal matrix $T = \mathrm{tridiag}(1, -2, 1)$ acting on the interior points (the same matrix as in the BVP notebook), the **θ-method** is

$$
\left(I - \theta\lambda T\right)\mathbf{u}^{n+1} = \left(I + (1 - \theta)\lambda T\right)\mathbf{u}^{n} + \lambda\,\mathbf{b},
$$

where $\mathbf{b}$ holds the (fixed) boundary values: $u_0$ in its first entry, $u_{N-1}$ in its last, and zeros elsewhere.

| θ | Method | Accuracy in time | Stability |
|---|---|---|---|
| 0 | FTCS (explicit) | $O(\Delta t)$ | $\lambda \le 1/2$ |
| 1 | Backward Euler, BTCS (implicit) | $O(\Delta t)$ | always |
| 1/2 | [Crank–Nicolson](https://en.wikipedia.org/wiki/Crank%E2%80%93Nicolson_method) | $O(\Delta t^2)$ | always |

Each step now requires solving a tridiagonal system. The matrix does not change between steps, so we **factorize it once** with `spla.factorized` (an LU decomposition) and reuse the factors every step. For a tridiagonal matrix this costs $O(N)$ per step, the same order as FTCS.

```{figure} https://upload.wikimedia.org/wikipedia/commons/thumb/1/1e/Crank-Nicolson-stencil.svg/250px-Crank-Nicolson-stencil.svg.png
:width: 25%
:align: center
Crank–Nicolson stencil: three new and three old values are coupled.
```

In [ ]:
def theta_stepper(N, lam, theta):
    """Return a function step(u) that advances u by one θ-method step (fixed end values)."""
    n = N - 2
    T = sp.diags([1.0, -2.0, 1.0], [-1, 0, 1], shape=(n, n))
    I = sp.identity(n)
    solve_A = spla.factorized((I - theta*lam*T).tocsc())    # LU factorization, done once
    B = (I + (1 - theta)*lam*T).tocsr()

    def step(u):
        b = B @ u[1:-1]
        b[0] += lam*u[0]            # boundary contributions (fixed in time)
        b[-1] += lam*u[-1]
        u_new = u.copy()
        u_new[1:-1] = solve_A(b)
        return u_new
    return step

### Large time steps

With $\lambda = 5$ (ten times the explicit limit), FTCS explodes while backward Euler and Crank–Nicolson stay stable and close to the exact decay of the $\sin(\pi x)$ mode.

In [ ]:
N, alpha, lam, t_end = 41, 1.0, 5.0, 0.1
x = np.linspace(0, 1, N)
h = x[1] - x[0]
dt = lam*h**2/alpha
nsteps = int(round(t_end/dt))
exact = np.sin(np.pi*x)*np.exp(-alpha*np.pi**2*nsteps*dt)
fig, ax = plt.subplots(figsize=(6, 4))
for theta, name in [(0.0, "FTCS"), (1.0, "Backward Euler"), (0.5, "Crank-Nicolson")]:
    step = theta_stepper(N, lam, theta)
    u = np.sin(np.pi*x)
    for n in range(nsteps):
        u = step(u)
    err = np.max(np.abs(u - exact))
    if theta == 0:
        print(f"{name:15s}: max |u| = {np.max(np.abs(u)):.2e} (unstable)")
    else:
        print(f"{name:15s}: max error = {err:.2e}")
        ax.plot(x, u, "o", ms=4, label=f"{name}")
ax.plot(x, exact, "k-", lw=1, label="exact")
ax.set(xlabel="$x$", ylabel="$u$", title=f"{nsteps} steps with $\\lambda$={lam} (t={nsteps*dt:.3f})")
ax.legend()
plt.show()

### "Stable" is not "accurate"

Stability only means that errors do not grow. With a large $\lambda$ and a sharp initial condition (a step), Crank–Nicolson shows **spurious oscillations** that decay very slowly: its amplification factor for short waves is close to $-1$. Backward Euler damps them strongly (it is smooth but only first order). A practical recipe is to take a few backward-Euler steps first and then continue with Crank–Nicolson.

In [ ]:
N, lam = 81, 20.0
x = np.linspace(0, 1, N)
u0 = np.where((x > 0.25) & (x < 0.75), 1.0, 0.0)
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
reference = u0.copy()
fine = theta_stepper(N, 0.5, 1.0)                     # reference: many small implicit steps
for n in range(40*3):
    reference = fine(reference)
for ax, (theta, name) in zip(axes, [(1.0, "Backward Euler"), (0.5, "Crank-Nicolson")]):
    step = theta_stepper(N, lam, theta)
    u = u0.copy()
    for n in range(3):
        u = step(u)
    ax.plot(x, u0, "k:", lw=1, label="initial")
    ax.plot(x, reference, "k-", lw=1, label="reference")
    ax.plot(x, u, "o-", ms=3, label=f"{name}, 3 steps, $\\lambda$={lam}")
    ax.set(xlabel="$x$", title=name)
    ax.legend(fontsize=8)
axes[0].set_ylabel("$u$")
plt.tight_layout()
plt.show()

```{exercise} Order of accuracy in time
:label: ex-heat-time-order
Measure how the time-stepping error of backward Euler and Crank–Nicolson decreases with $\Delta t$, for $u(x, 0) = \sin(\pi x)$ ($N = 101$, $\alpha = 1$) integrated to $t = 0.1$. To remove the spatial error we compare with the exact solution of the space-discretized equations, $\sin(\pi x)\,e^{-\alpha\mu t}$ with $\mu = \frac{4}{h^2}\sin^2\!\left(\frac{\pi h}{2}\right)$ (already in the function).
1. Complete `time_error(theta, dt)`: build `step = theta_stepper(N, alpha*dt/h**2, theta)`, apply it `round(t_end/dt)` times to the initial condition, and return the maximum error.
2. Run the next cell, which plots the error vs $\Delta t$ in log-log scale. Read the slopes. Which method would you choose for an accurate long simulation?

```

In [ ]:
def time_error(theta, dt, N=101, alpha=1.0, t_end=0.1):
    """Max error at t_end of the θ-method for u(x, 0) = sin(πx), without spatial error."""
    x = np.linspace(0, 1, N)
    h = x[1] - x[0]
    mu = 4/h**2*np.sin(np.pi*h/2)**2
    exact = np.sin(np.pi*x)*np.exp(-alpha*mu*t_end)
    # YOUR CODE HERE
    pass

In [ ]:
# Given: run it after completing time_error
dts = np.array([0.02, 0.01, 0.005, 0.0025, 0.00125])
fig, ax = plt.subplots(figsize=(5.5, 4))
for theta, name in [(1.0, "Backward Euler"), (0.5, "Crank-Nicolson")]:
    errs = [time_error(theta, dt) for dt in dts]
    slope = np.polyfit(np.log(dts), np.log(errs), 1)[0]
    ax.loglog(dts, errs, "o-", label=f"{name} (slope {slope:.2f})")
ax.set_xticks(dts, [f"{d:g}" for d in dts])
ax.minorticks_off()
ax.set(xlabel=r"$\Delta t$", ylabel="max error at $t=0.1$", title="Time-stepping error")
ax.legend()
save_example("heat_time_order")
plt.show()

In [ ]:
for name in ['heat_time_order']:
    if os.path.exists(f"fig/{name}.png"):
        display(Image(filename=f"fig/{name}.png"))

## Two dimensions

In 2D, $\partial_t u = \alpha(\partial_x^2 u + \partial_y^2 u)$, and FTCS with $h_x = h_y = h$ uses the five-point stencil of the relaxation notebook:

$$
u_{i,j}^{n+1} = u_{i,j}^n + \lambda\left(u_{i+1,j}^n + u_{i-1,j}^n + u_{i,j+1}^n + u_{i,j-1}^n - 4u_{i,j}^n\right), \qquad \lambda \le \frac14 .
$$

(For different spacings the condition is $\alpha\Delta t\left(1/h_x^2 + 1/h_y^2\right) \le 1/2$.) Below, a hot Gaussian spot spreads in a plate whose edges are held at $u = 0$.

In [ ]:
def ftcs2d(u, lam):
    """One FTCS step of the 2D heat equation (edges unchanged)."""
    u_new = u.copy()
    u_new[1:-1, 1:-1] = u[1:-1, 1:-1] + lam*(u[2:, 1:-1] + u[:-2, 1:-1] + u[1:-1, 2:] + u[1:-1, :-2]
                                              - 4*u[1:-1, 1:-1])
    return u_new


N, alpha, lam = 61, 1.0, 0.2
x = np.linspace(0, 1, N)
h = x[1] - x[0]
dt = lam*h**2/alpha
XX, YY = np.meshgrid(x, x, indexing="ij")
u = np.exp(-((XX - 0.5)**2 + (YY - 0.5)**2)/(2*0.08**2))
u[0, :] = u[-1, :] = u[:, 0] = u[:, -1] = 0.0
frames, times = [u.copy()], [0.0]
nsteps = 800
for n in range(1, nsteps + 1):
    u = ftcs2d(u, lam)
    if n % 20 == 0:
        frames.append(u.copy())
        times.append(n*dt)
animate_2d(frames, times)

Implicit methods in 2D need the matrix of the 2D Laplacian. Three ideas are enough to build it:

- **Flatten the unknowns.** The $n\times n$ interior values `u[1:-1, 1:-1]` become one vector with `.ravel()`, row after row: point $(i, j)$ goes to position $k = i\,n + j$. The $y$-neighbours $(i, j\pm1)$ sit at $k\pm1$ and the $x$-neighbours $(i\pm1, j)$ at $k\pm n$.
- **Kronecker products.** `sp.kron(P, Q)` replaces each entry $p_{ab}$ of $P$ by the block $p_{ab}Q$. With the 1D matrix $T$ and the identity $I$, `sp.kron(T, I)` couples $k$ with $k\pm n$ (second difference in $x$) and `sp.kron(I, T)` couples $k$ with $k\pm1$ inside each row (second difference in $y$). Their sum $A = T\otimes I + I\otimes T$ is the five-point stencil: $-4$ on the diagonal and $1$ for each neighbour (see the small example below).
- **Sparse format.** `factorized` (and `spsolve`) need the *compressed column* format, so we convert once with `.tocsc()`.

The matrix is no longer tridiagonal but it is still sparse, so `spla.factorized` works well for moderate grids. For large 3D problems, *ADI* methods ([alternating-direction implicit](https://en.wikipedia.org/wiki/Alternating-direction_implicit_method)) split each step into 1D tridiagonal solves.

In [ ]:
n = 3                                                  # 3 x 3 interior points -> 9 unknowns
T = sp.diags([1.0, -2.0, 1.0], [-1, 0, 1], shape=(n, n))
I = sp.identity(n)
A = sp.kron(T, I) + sp.kron(I, T)
print("T =\n", T.toarray().astype(int))
print("\nA = kron(T, I) + kron(I, T) =\n", A.toarray().astype(int))

```{exercise} Crank–Nicolson in 2D
:label: ex-heat-cn2d
Solve the same hot-spot problem ($N = 61$, $\alpha = 1$, edges at 0) up to $t = 0.02$ with Crank–Nicolson and $\lambda = 2$ (ten times the explicit step).
1. Complete `cn2d_stepper(N, lam)`, the 2D version of `theta_stepper` with $\theta = 1/2$: the matrix `A` is already built; write the two lines that create `solve_M` (factorize `Id - 0.5*lam*A`) and `B = Id + 0.5*lam*A`.
2. Run the next cell, which compares the result with FTCS ($\lambda = 0.2$) at the same final time. How many steps did each method take? Is the difference significant?
```

In [ ]:
def cn2d_stepper(N, lam):
    """Return step(u) that advances the 2D field u by one Crank-Nicolson step (edges at 0)."""
    n = N - 2
    T = sp.diags([1.0, -2.0, 1.0], [-1, 0, 1], shape=(n, n))
    I = sp.identity(n)
    A = sp.kron(T, I) + sp.kron(I, T)            # 2D Laplacian (times h^2) on the n*n interior points
    Id = sp.identity(n*n)
    # YOUR CODE HERE
    pass

    def step(u):
        u_new = u.copy()
        u_new[1:-1, 1:-1] = solve_M(B @ u[1:-1, 1:-1].ravel()).reshape(n, n)
        return u_new
    return step

In [ ]:
# Given: run it after completing cn2d_stepper
N, alpha, t_end = 61, 1.0, 0.02
x = np.linspace(0, 1, N)
h = x[1] - x[0]
XX, YY = np.meshgrid(x, x, indexing="ij")
u0 = np.exp(-((XX - 0.5)**2 + (YY - 0.5)**2)/(2*0.08**2))
u0[0, :] = u0[-1, :] = u0[:, 0] = u0[:, -1] = 0.0

results = {}
for name, lam in [("Crank-Nicolson", 2.0), ("FTCS", 0.2)]:
    nsteps = int(round(t_end/(lam*h**2/alpha)))
    lam = alpha*(t_end/nsteps)/h**2                  # land exactly on t_end
    step = cn2d_stepper(N, lam) if name == "Crank-Nicolson" else (lambda u, lam=lam: ftcs2d(u, lam))
    u = u0.copy()
    for k in range(nsteps):
        u = step(u)
    results[f"{name} ({nsteps} steps)"] = u

(name1, u_cn), (name2, u_ex) = results.items()
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, field, title, cmap in [(axes[0], u_cn, name1, "inferno"), (axes[1], u_ex, name2, "inferno"),
                               (axes[2], u_cn - u_ex, "difference CN - FTCS", "RdBu_r")]:
    im = ax.imshow(field.T, origin="lower", extent=(0, 1, 0, 1), cmap=cmap)
    plt.colorbar(im, ax=ax)
    ax.set(xlabel="$x$", ylabel="$y$", title=title)
plt.tight_layout()
save_example("heat_cn2d")
plt.show()

In [ ]:
for name in ['heat_cn2d']:
    if os.path.exists(f"fig/{name}.png"):
        display(Image(filename=f"fig/{name}.png"))

## Summary

| Method | Stability | Order (time, space) | Cost per step | Use it when |
|---|---|---|---|---|
| FTCS (explicit) | $\lambda \le 1/2$ (1D), $\le 1/4$ (2D) | $O(\Delta t, h^2)$ | $O(N)$, trivial to code | small grids, short times, quick tests |
| Backward Euler | always | $O(\Delta t, h^2)$ | one sparse solve (factorize once) | stiff problems, robustness, sharp initial data |
| Crank–Nicolson | always | $O(\Delta t^2, h^2)$ | one sparse solve (factorize once) | accurate long runs (start with a few BE steps) |

Key ideas:
- FTCS is the Jacobi sweep in time; its stability limit $\Delta t \le h^2/(2\alpha)$ makes fine grids very expensive.
- Von Neumann analysis predicts exactly which mode blows up (the saw-tooth) and how fast.
- Always validate: exact modes $\sin(\pi x)e^{-\alpha\pi^2 t}$, steady states, conservation of heat, the diffusion law $\sigma^2 = \sigma_0^2 + 2\alpha t$.
- Insulated ends: ghost point $u_{-1} = u_1$. Sources and sinks are extra terms in the update.
- Implicit schemes cost one sparse solve per step but allow large steps; stability does not guarantee accuracy.

## Further exercises
1. **Variable conductivity.** Solve $\partial_t u = \partial_x\big(\alpha(x)\,\partial_x u\big)$ for a rod made of two materials ($\alpha = 1$ on the left half, $\alpha = 0.1$ on the right). Use $\alpha$ at the half points $x_{i\pm1/2}$ in the stencil.
2. **Periodic ring.** Solve the heat equation on a ring (periodic boundaries) using `np.roll` and check that the total heat is conserved.
3. **Fisher–KPP equation.** Add the logistic reaction term $r\,u(1 - u)$ and observe a travelling front (compare its speed with $2\sqrt{r\alpha}$). See also the reaction–diffusion presets of [VisualPDE](https://visualpde.com/explore.html).
4. **ADI.** Implement the Peaceman–Rachford ADI scheme for the 2D problem, using only tridiagonal solves.
5. **Insulated plate in 2D.** Repeat the insulated-rod exercise in 2D with all four edges insulated.